In [ ]:
# allows update of external libraries without need to reload package
%load_ext autoreload
%autoreload 2

# Layered data validation and processing demonstration

## Summary

Demonstrate a workflow of a layered data validation and processing of WTG data.

## Results

### Key results

- Basic structure is a medaillon architecture w/ layers raw, bronze, silver and gold providing tracking of results.
- Demonstration includes raw and bronze layer diagnostics and a transformation built on raw layer results.
  - Example: Messed synthetic data.
- Open to extensions and refinements regarding:
  - Validation rules.
  - Tracking.
  - Failure/passed logic and their aggregations.

### Details

- Core functionality is a tracking, logging and reporting mechanism of the results, currently as structured summary.
  - Possibility for a reporting system open.
- Validations are configurable and versionable.
- Layers (concept):
  - Raw:
    - Formal diagnostics for a standardisation.
  - Bronze:
    - Domain-related diagnostics for outlier detection.
  - Silver:
    - Gap filling.
  - Gold:
    - Advanced stuff.
- Layers validate:
  - Raw:
    - `required_variable`: Check if required variables are present in the data.
    - `temporal_attributes`: Detect temporal properties of the timeseries: Start, end resolution, ...
    - `data_gaps`: Detect and create summary statistics of gaps in timestamps.
    - `availability`: Determine the availability.
  - Bronze:
    - `ranges`: Validate provided ranges for variables.
    - `essential_ranges`: Detect the essential minimum and maximum of listed variables.
    - `curtailments_power`: Detect curtailments.


## Remarks

- Modules containing rules need to be imported. This ensures all `ValidationRules` are registered.


## Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats
import scipy.signal
import yaml

import ergaleiothiki
from phoibe.layered.application.config import ValidationConfig
from phoibe.layered.application.context import ValidationContext
from phoibe.layered.application.factory import ValidatorFactory, RuleRegistry
from phoibe.layered.infrastructure.io import YAMLReportRepository
from phoibe.layered.core.entities import RuleExecutionResult, Status, Severity

import phoibe.synthetic_data.turbine
from phoibe.synthetic_data.turbine import MessUpPipeline, create_default_messup_pipeline

import phoibe.layered.rules.rules_index
import phoibe.layered.rules.rules_columns
import phoibe.layered.rules.rules_values
import phoibe.layered.rules.rules_power

## Provision

### Layer rule configuration

The configuration requires:
- Regex rules to match variable names to column keys.
- Validation rules including their configuration.

Typically, they will be stored in yaml files and read from there.

In [ ]:
VARIABLE_PATTERNS = {
    "power_kw": [r"power", r"production", r"power_kw"],
    "wind_speed": [r"wind.*speed", r"wind.*speed.*avg"],
    "wind_speed_max": [r"wind.*speed.*max"],
    "datetime": [r"datetime", r"timestamp"],
    "date": [r"date"],
    "time": [r"time"],
}
RULES_RAW = [
    {"name": "required_variable", "params": {"variable_name": "power_kw"}},
    {"name": "temporal_attributes", "params": {}},
    {"name": "data_gaps", "params": {"good_threshold": 0.03, "acceptable_threshold": 0.1}},
    {"name": "availability", "params": {"good_threshold": 0.9, "acceptable_threshold": 0.8, "locale": "de_DE"}},
]
RULES_BRONZE = [
    {"name": "ranges", "params": {"variable_ranges": {"wind_speed": (0, 60)}}},
    {"name": "essential_range", "params": {"variable_names": ["wind_speed", "power_kw", "rotor_speed", "pitch_angle"]}},
    {"name": "curtailments_power", "params": {}},
]

config_raw = ValidationConfig(layer_name="raw", variable_patterns=VARIABLE_PATTERNS, rules=RULES_RAW)
config_bronze = ValidationConfig(layer_name="bronze", variable_patterns=VARIABLE_PATTERNS, rules=RULES_BRONZE)

In [ ]:
RuleRegistry().list_rules()

### Synthetic data

This creates synthetic data and messes them.

In [ ]:
TIME = phoibe.synthetic_data.turbine.Time(start="2027-01-01", freq="10min", periods=6 * 24 * 365)
A, k = 9.7, 2.3
df = phoibe.synthetic_data.turbine.generate_wtg_scada(
    A=A, k=k, time=TIME, wtg_type=phoibe.synthetic_data.turbine.DEFAULT_WTG
)

mess_up_pipeline = phoibe.synthetic_data.turbine.create_default_messup_pipeline(incidence=23, level=13)
data = mess_up_pipeline.apply(df=df)

display(pd.concat([df.mean(), data.mean()], keys=["synthetic", "messed"], axis=1))
display(pd.concat([df.describe(), data.describe()], keys=["synthetic", "messed"], axis=1))

In [ ]:
data = data.reset_index()
data

## Validation-Transformation

### Validation of layer raw

Apply the raw layer validation rules and create the report.

In [ ]:
validator_raw = ValidatorFactory().create_from_memory(config=config_raw, data=data)
report_raw = validator_raw.validate(file_path=".", turbine_id="WEA 01")

### Reporting

Peek into the report.

In [ ]:
report_raw.rule_execution_results

In [ ]:
YAMLReportRepository().save(reported=[report_raw], output_path="reports/report_raw.yaml")

with open("reports/report_raw.yaml", "r") as stream:
    print(yaml.safe_dump(yaml.safe_load(stream), explicit_start=True))

In [ ]:
context = ValidationContext(
    detected_variables=validator_raw.variable_detector.detect(data), turbine_id="WEA 02", layer_name="raw"
)
context.detected_variables

### Transform layer raw

The following transformation means to normalise the data. This includes renaming the detected columns, and removing duplicates, providing timezones (this part is skipped here for now).

In [ ]:
key_mapping = {
    value: key for key, value in context.detected_variables.items() if key in ["datetime", "wind_speed", "power_kw"]
}
data_bronze = data.rename(columns=key_mapping)
plt.scatter(data_bronze["wind_speed"], data_bronze["power_kw"])

### Validation of layer bronze - experimental stuff for validation rules

In [ ]:
def high_density_range(data: np.typing.NDArray[np.floating], mass: float = 0.99) -> tuple[float, float]:
    x = np.sort(data)
    n = len(x)
    k = int(np.floor(mass * n))

    widths = x[k:] - x[: n - k]
    i = np.argmin(widths)

    return float(x[i]), float(x[i + k])


def hdi_length(
    data: np.typing.NDArray[np.floating], grid: np.typing.NDArray[np.floating] = np.linspace(0.98, 0.999, num=20)
) -> tuple[np.typing.NDArray[np.floating], list[float]]:
    length = []
    for mass in grid:
        data_min, data_max = high_density_range(data, mass=mass)
        length.append(data_max - data_min)
    return grid, length

In [ ]:
high_density_range(data_bronze.loc[:, "power_kw"], mass=0.995)

In [ ]:
power_kde = scipy.stats.gaussian_kde(data_bronze.loc[:, "power_kw"])
print(power_kde.covariance, power_kde.factor, power_kde.silverman_factor())
extent_min = data_bronze.loc[:, "power_kw"].min()
extent_max = data_bronze.loc[:, "power_kw"].max()
t = np.linspace(extent_min, extent_max, num=1000)
p = power_kde(t)
p = p / p.sum()

figure, axes = plt.subplots(2, 2)
axes[0, 0].plot(t, p)

dp = (p[1:] - p[:-1]) / (t[1:] - t[:-1])
axes[0, 1].plot(t[1:], dp)

peak_indices, peak_properties = scipy.signal.find_peaks(
    p, height=(None, None), threshold=(None, None), prominence=(None, None), width=(None, None)
)
axes[0, 0].scatter(0.5 * t[peak_indices - 1] + 0.5 * t[peak_indices], p[peak_indices], c="r")
display(t[peak_indices])
display(peak_properties)

ps = np.cumsum(p)
axes[1, 0].plot(t, ps)

p_min, p_max = high_density_range(data_bronze.loc[:, "power_kw"], mass=0.995)
w_min, w_max = high_density_range(data_bronze.loc[:, "wind_speed"], mass=0.995)
axes[0, 0].scatter((p_min, p_max), (0, 0), c="g")

s, h = hdi_length(data_bronze.loc[:, "wind_speed"])
axes[1, 1].plot(s, h)

In [ ]:
# Goal: Determine curtailments.
mask_curtailments = data_bronze.loc[:, "wind_speed"] > 14
data_bronze.loc[mask_curtailments, "power_kw"].hist(bins=60, log=True)
# heuristic: for wind_speeds > 15 m/s, determine clusters of powers in histogram
# by comparing bin occupations - there should be a gap of magnitude larger than occupation in bulk.
occupation, value = np.histogram(data_bronze.loc[mask_curtailments, "power_kw"], bins=100)
counts = pd.Series(index=(value[1:] + value[:-1]) / 2, data=occupation).sort_values(ascending=False)
display(-counts.diff() / counts)

# heuristic: for wind_speeds > 15 m/s, determine peaks of kernel density estimates.

power_kde = scipy.stats.gaussian_kde(data_bronze.loc[mask_curtailments, "power_kw"])
extent_min = data_bronze.loc[mask_curtailments, "power_kw"].min()
extent_max = data_bronze.loc[mask_curtailments, "power_kw"].max()
t = np.linspace(extent_min, extent_max, num=1000)
p = power_kde(t)

figure, axes = plt.subplots(1, 2)
axes[0].plot(t, p)

dp = (p[1:] - p[:-1]) / (t[1:] - t[:-1])
axes[1].plot(t[1:], dp)

peak_indices, peak_values = scipy.signal.find_peaks(
    p, height=(None, None), threshold=(None, None), prominence=(None, None), width=(None, None)
)
axes[0].scatter(0.5 * t[peak_indices - 1] + 0.5 * t[peak_indices], p[peak_indices], c="r")
display(t[peak_indices])
display(peak_values)

In [ ]:
peak_indices
list(t[peak_indices])

### Validation of layer bronze

In [ ]:
validator_bronze = ValidatorFactory().create_from_memory(config=config_bronze, data=data_bronze)
report_bronze = validator_bronze.validate(file_path=".", turbine_id="WEA 01")

In [ ]:
YAMLReportRepository().save(reported=[report_bronze], output_path="reports/report_bronze.yaml")

with open("reports/report_bronze.yaml", "r") as stream:
    print(yaml.safe_dump(yaml.safe_load(stream), explicit_start=True))